In [2]:
# Import Libraries
import pandas as pd


In [72]:
#Date Reading and Check

df = pd.read_csv("Online Retail Data/online_retail.csv", encoding = "ISO-8859-1")
df.head()
df.info()
df.describe()

<class 'pandas.DataFrame'>
RangeIndex: 541909 entries, 0 to 541908
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype  
---  ------       --------------   -----  
 0   InvoiceNo    541909 non-null  str    
 1   StockCode    541909 non-null  str    
 2   Description  540455 non-null  str    
 3   Quantity     541909 non-null  int64  
 4   InvoiceDate  541909 non-null  str    
 5   UnitPrice    541909 non-null  float64
 6   CustomerID   406829 non-null  float64
 7   Country      541909 non-null  str    
dtypes: float64(2), int64(1), str(5)
memory usage: 33.1 MB


,Quantity,UnitPrice,CustomerID
count,541909.000000,541909.000000,406829.000000
mean,9.552250,4.611114,15287.690570
std,218.081158,96.759853,1713.600303
min,-80995.000000,-11062.060000,12346.000000
25%,1.000000,1.250000,13953.000000
50%,3.000000,2.080000,15152.000000
75%,10.000000,4.130000,16791.000000
max,80995.000000,38970.000000,18287.000000


In [13]:
#Converting InvoiceDate from String to DateTime type
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])

In [73]:
#Checking Null values
df.isnull().sum()

InvoiceNo           0
StockCode           0
Description      1454
Quantity            0
InvoiceDate         0
UnitPrice           0
CustomerID     135080
Country             0
dtype: int64

In [74]:
#Droping missing values from CustomerId

df =  df.dropna(subset=['CustomerID'])

In [75]:
#Dropping Duplicate Values

dr = df.drop_duplicates()

In [52]:
#Handling negative and zero values

df[df['Quantity'] <= 0]
df[df['UnitPrice'] <= 0]

df = df[(df['Quantity'] > 0) & (df['UnitPrice'] > 0)]

In [53]:
#Adding new columnn as TotalPrice

df['TotalPrice'] = df['UnitPrice'] * df['Quantity']

In [54]:
#Converting CustomerID to Int

df['CustomerID'] = df['CustomerID'].astype(int)

In [71]:
#Basic sanity check

# df.shape
df.describe()
# df['CustomerID'].nunique()
# df['InvoiceNo'].nunique()

,Quantity,UnitPrice,CustomerID,TotalPrice
count,397884.000000,397884.000000,397884.000000,397884.000000
mean,12.988238,3.116488,15294.423453,22.397000
std,179.331775,22.097877,1713.141560,309.071041
min,1.000000,0.001000,12346.000000,0.001000
25%,2.000000,1.250000,13969.000000,4.680000
50%,6.000000,1.950000,15159.000000,11.800000
75%,12.000000,3.750000,16795.000000,19.800000
max,80995.000000,8142.750000,18287.000000,168469.600000


In [66]:
#Exporting cleaned data CSV

df.to_csv("cleaned_retail_data.csv", index=False)

In [130]:
#=================
# Load Clean Data
#=================

df = pd.read_csv("clean_data/cleaned_retail_data.csv")
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])

In [131]:
#Null value check

df.isnull().sum()

InvoiceNo      0
StockCode      0
Description    0
Quantity       0
InvoiceDate    0
UnitPrice      0
CustomerID     0
Country        0
TotalPrice     0
dtype: int64

In [132]:
#Setting reference date
snapshot_date = df['InvoiceDate'].max() + pd.Timedelta(days=1)

#Grouping by customer
customer_df = df.groupby('CustomerID').agg({
    'InvoiceDate' : ['min', 'max'],
    'InvoiceNo' : 'nunique',
    'TotalPrice' : 'sum'
})

#Flatten column names
customer_df.columns = ['FirstPurchase','LastPurchase','TotalOrders','TotalSpent']

customer_df = customer_df.reset_index()

In [95]:
customer_df.head(20)

,CustomerID,FirstPurchase,LastPurchase,TotalOrders,TotalSpent
0,12346,2011-01-18 10:01:00,2011-01-18 10:01:00,1,77183.60
1,12347,2010-12-07 14:57:00,2011-12-07 15:52:00,7,4310.00
2,12348,2010-12-16 19:09:00,2011-09-25 13:13:00,4,1797.24
3,12349,2011-11-21 09:51:00,2011-11-21 09:51:00,1,1757.55
4,12350,2011-02-02 16:01:00,2011-02-02 16:01:00,1,334.40
5,12352,2011-02-16 12:33:00,2011-11-03 14:37:00,8,2506.04
6,12353,2011-05-19 17:47:00,2011-05-19 17:47:00,1,89.00
7,12354,2011-04-21 13:11:00,2011-04-21 13:11:00,1,1079.40
8,12355,2011-05-09 13:49:00,2011-05-09 13:49:00,1,459.40
9,12356,2011-01-18 09:50:00,2011-11-17 08:40:00,3,2811.43


In [133]:
#RFM
customer_df['Recency'] = (snapshot_date - customer_df['LastPurchase']).dt.days
customer_df['Frequency'] = customer_df['TotalOrders']
customer_df['Monetary'] = customer_df['TotalSpent']

In [134]:
#Churn 

churn_threshold = 90
customer_df['IsChurned'] = customer_df['Recency'] > churn_threshold

In [135]:
customer_df['IsChurned'].value_counts()
# customer_df.groupby('IsChurned')['monetary'].sum()

IsChurned
False    2889
True     1449
Name: count, dtype: int64

In [136]:
customer_df['segment'] = pd.qcut(customer_df['Monetary'], q=3, labels=['Low', 'Medium', 'High'])

customer_df.groupby(['segment', 'IsChurned']).size()

# customer_df.groupby(['segment', 'IsChurned'])['Monetary'].sum()

segment  IsChurned
Low      False         653
         True          793
Medium   False         961
         True          485
High     False        1275
         True          171
dtype: int64

In [137]:
customer_df.to_csv("customer_analysis.csv", index=False)